# SOXL局部净价值的既有风险状态跨年重叠

只读2018–2022原架构Qlib训练标签与09:46原风险状态，严格按预冻精确状态和上限受限标志比较2020–2021与2022。仅归因，不拟合或改动作。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_soxl_local_value_risk_state_overlap_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['SOXL_card'];b=root/s['reference_card']
assert sha(a/'summary.json')==s['SOXL_summary_sha256'] and sha(b/'summary.json')==s['reference_summary_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['SOXL_NAS_sha256']
assert sb['nas_readback']['verified'] and sb['recorder_id']==s['reference_recorder_id']
assert sha(a/'qlib_recorder'/s['SOXL_experiment_id']/s['SOXL_recorder_id']/'artifacts/analysis/labels')==s['SOXL_labels_sha256']
assert sha(b/'qlib_recorder'/s['reference_experiment_id']/s['reference_recorder_id']/'artifacts/paths/reference_0')==s['reference_path_sha256']
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['SOXL_experiment_id'],recorder_id=s['SOXL_recorder_id']);labels=rec.load_object('analysis/labels')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['reference_experiment_id'],recorder_id=s['reference_recorder_id']);bundle=rec.load_object('paths/reference_0')
path=bundle['replay'];assert len(labels)==len(path)==1061
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'))
ld=pd.DatetimeIndex(labels.trade_date);pdts=pd.DatetimeIndex(path.trade_date)
assert ld.equals(pdts) and ld.equals(cal)
assert pd.DatetimeIndex(labels.signal_date).equals(pd.DatetimeIndex(path.signal_date))
assert labels.soxl_local_fee_net_advantage.notna().sum()==688 and set(path.risk_state.dropna().astype(int).unique()).issubset({0,1,2,3})
x=pd.DataFrame({'trade_date':ld,'year':ld.year,'quarter':ld.quarter,'risk_state':path.risk_state.to_numpy(int),'cap_binding':labels.cap_binding.to_numpy(bool),
                'current_beta_cap':path.current_beta_cap.to_numpy(float),'portfolio_drawdown':path.portfolio_drawdown.to_numpy(float),
                'eligible':labels.eligible.to_numpy(bool),'label':labels.soxl_local_fee_net_advantage.to_numpy(float)})
assert x.current_beta_cap.notna().all() and x.portfolio_drawdown.notna().all()
assert x.loc[x.label.notna(),'eligible'].all() and x.loc[x.label.notna(),'risk_state'].isin([0,1,2,3]).all()
valid=x[x.label.notna()].copy()
def stats(g):
 z=g.label.dropna()
 return {'sessions':len(g),'mature':len(z),'mean':float(z.mean()) if len(z) else None,'median':float(z.median()) if len(z) else None,
         'positive_fraction':float((z>0).mean()) if len(z) else None,
         'median_cap':float(g.current_beta_cap.median()) if len(g) else None,
         'median_drawdown':float(g.portfolio_drawdown.median()) if len(g) else None}
annual=[{'year':int(year),'partial_endpoint':year==2018,**stats(g)} for year,g in x.groupby('year',sort=True)]
quarter=[{'year':int(year),'quarter':int(q),**stats(g)} for (year,q),g in x.groupby(['year','quarter'],sort=True)]
assert len(annual)==5 and len(quarter)==17 and sum(a['mature'] for a in annual)==688
cells=[]
for state in range(4):
 for binding in (False,True):
  g=valid[(valid.risk_state==state)&(valid.cap_binding==binding)]
  earlier=g[g.year.isin([2020,2021])];later=g[g.year==2022]
  row={'risk_state':state,'cap_binding':binding,'earlier_2020_2021':stats(earlier),'later_2022':stats(later),
       'matched':bool(len(earlier)>=10 and len(later)>=10)}
  if row['matched']:row['later_minus_earlier_mean']=float(later.label.mean()-earlier.label.mean())
  cells.append(row)
matched=[r for r in cells if r['matched']];matched_2022=sum(r['later_2022']['mature'] for r in matched)
all_2022=int((valid.year==2022).sum())
weighted_diff=(float(np.average([r['later_minus_earlier_mean'] for r in matched],weights=[r['later_2022']['mature'] for r in matched])) if matched else None)
result={'status':'read_only_SOXL_local_value_predecision_risk_state_overlap_no_policy','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'labels':s['SOXL_recorder_id'],'reference':s['reference_recorder_id']},
        'full_sessions':len(x),'mature_labels':len(valid),'annual':annual,'quarter':quarter,'cells':cells,'matched_cells':len(matched),
        'matched_2022_labels':matched_2022,'full_2022_labels':all_2022,'matched_2022_coverage':matched_2022/all_2022,
        'weighted_later_minus_earlier_mean':weighted_diff,
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Same-state era comparison uses historical outcomes and is not a causal or tradable regime switch',
                  '2018-2022 original architecture training local labels are not a complete exact-f696 funded account',
                  'The 2022 era and state cells are reused and not independent forward validation']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'matched_cells':len(matched),'coverage':result['matched_2022_coverage'],'weighted_diff':weighted_diff,
       'cells':[(r['risk_state'],r['cap_binding'],r['earlier_2020_2021']['mature'],r['later_2022']['mature'],r.get('later_minus_earlier_mean')) for r in cells]})


{'matched_cells': 4, 'coverage': 0.9919028340080972, 'weighted_diff': -0.00047310282115734013, 'cells': [(0, False, 22, 0, None), (0, True, 26, 10, -0.002536673298995563), (1, False, 0, 0, None), (1, True, 48, 40, -0.0005792222893720191), (2, False, 0, 0, None), (2, True, 116, 100, -5.5220644823388766e-05), (3, False, 24, 2, None), (3, True, 48, 95, -0.0006510789698565587)]}


## 中文结论

**更正（2026-10-24）：** 原688个标签中148个未受限动作超过30% SOXL上限，不能用于全体统计。本文四个匹配格子均为合法受限动作，2022覆盖分母应为245/245，而非旧245/247；格内差异不变。详见[权重合法性审计](../stage1_soxl_local_action_max_weight_legality_audit_20261024/conclusion.md)。

# 相同风险状态下，SOXL局部价值仍发生跨年反转

只读原架构2018–2022 Qlib训练账本的1061个交易日、688个成熟现金转SOXL含Tiered费局部标签和原09:46风险状态。逐件核对Recorder工件SHA、NAS定向回读、Qlib日历及信号/交易日对齐；保留部分2018、完整2019–2022和17个季度。按计划只使用事前已知的精确 `risk_state` 0–3 与上限受限标志，比较2020/2021合并期和2022；每期每格至少10个标签才匹配，未调阈值或拟合。

2022年的247个成熟标签中，**245个（99.2%）**落在四个与2020/2021可比的相同状态×受限格子，且四格的2022平均局部价值都更低。按各格2022样本数加权，2022减2020/2021的局部净价值差 **−0.00047310（约−4.73bp）**。受限格 `risk_state` 0/1/2/3 的2020/2021与2022样本数分别为26/10、48/40、116/100、48/95；差值分别约−25.37、−5.79、−0.55、−6.51bp。仅有2个2022未受限标签未匹配。各年、季度、格子均值/中位数/胜率与原beta上限、账户回撤中位数见 `result.json`。

这表明**风险状态编号加受限标志本身**不能稳定区分2022式现金保护与2020/2021式部署机会；它们高度重叠，不能据此复活旧状态开关或重复拟合等价的41输入SOXX上限释放接收器。格子内2022的账户回撤中位数较高，提示连续自有状态仍可能有信息，但这是事后区间差异；连续回撤的方向与跨年稳定性尚未检验。下个单一必要条件应先固定“回撤越深，部署局部价值越低”的事前方向，逐年检验，不从这四个未来均值挑阈值。决定 `descriptive_same_risk_states_cross_year_local_value_sign_shift_no_policy`。没有新拟合、资金账户或SOXX前向证据。
